In [ ]:
import torch, librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

name = "openai/whisper-small"
proc = WhisperProcessor.from_pretrained(name)
model = WhisperForConditionalGeneration.from_pretrained(name, torch_dtype=torch.float16).to("cuda").eval()

audio, _ = librosa.load("assignment1.wav", sr=16000, mono=True)
feats = proc(audio, sampling_rate=16000, return_tensors="pt").input_features.to("cuda", torch.float16)

with torch.no_grad():
    ids = model.generate(feats, language="ar", task="transcribe",
                         num_beams=5, do_sample=False, return_timestamps=False)
print(proc.batch_decode(ids, skip_special_tokens=True)[0])

In [ ]:
import time, torch
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

ATTN = ["q_proj", "v_proj"]
ALL  = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]

def profile(rank, targets, steps=6):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=2*rank,
                           target_modules=targets, lora_dropout=0.05, bias="none"))
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    x = torch.randn(1, 80, 3000, device="cuda")
    y = torch.randint(0, 50000, (1, 80), device="cuda")
    times = []
    for _ in range(steps):
        torch.cuda.synchronize(); t = time.time()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            loss = model(input_features=x, labels=y).loss
        loss.backward(); opt.step(); opt.zero_grad()
        torch.cuda.synchronize(); times.append(time.time() - t)
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"r={rank:>2} targets={len(targets)} peak={peak:.2f} GB  step={sum(times[1:])/len(times[1:]):.2f}s")
    del model, opt; torch.cuda.empty_cache()

for r in (8, 16, 32):
    for t in (ATTN, ALL):
        profile(r, t)